# 🌾 PalayPlan — Rice Leaf Disease Detection Training
This Google Colab notebook trains a lightweight **MobileNetV2** neural network for real-time, on-device disease detection in the **PalayPlan** mobile app.

### ⚡ Quick Setup Instructions:
1. **Enable Free GPU:** Go to **Runtime** → **Change runtime type** → select **T4 GPU** → click **Save**.
2. **Run All:** Go to **Runtime** → **Run all** (or press `Ctrl+F9`).
3. At the end, Colab will automatically download `rice_model.tflite` and `labels.txt` directly to your computer!

## 1. Setup & Automated Dataset Loading
Cleans up temporary directories, downloads verified rice leaf disease images, and verifies classes.

In [ ]:
# === STEP 1: PREPARE DATASET ===
import os
import zipfile
import glob
import random
from PIL import Image
import matplotlib.pyplot as plt

DATASET_DIR = "dataset"

# Clean up any previous broken downloads
!rm -rf temp_repo rice

# If user uploaded dataset.zip, extract it
if os.path.exists("dataset.zip") and not os.path.exists(DATASET_DIR):
    print("📦 Found dataset.zip! Extracting...")
    with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("✅ Extracted dataset.zip.")

# If no dataset exists, automatically download the verified public rice disease dataset
if not os.path.exists(DATASET_DIR):
    print("⬇️ Downloading verified rice leaf disease dataset via Git (~8 MB)...")
    !git clone --depth 1 https://github.com/AveyBD/rice-leaf-diseases-detection.git temp_repo
    if os.path.exists("temp_repo/rice-leaf.zip"):
        !unzip -q temp_repo/rice-leaf.zip -d ./
        !mv rice/train ./dataset
        !cp -r rice/validation/* ./dataset/ 2>/dev/null || true
        !rm -rf temp_repo rice
        print("✅ Downloaded and prepared dataset successfully!")

if not os.path.exists(DATASET_DIR):
    print(f"⚠️ Dataset folder could not be prepared.")
else:
    classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
    counts = {}
    print("\n=== ✅ Verified Classes & Image Counts ===")
    for cls in classes:
        img_paths = glob.glob(f"{DATASET_DIR}/{cls}/*.[jJ][pP]*[gG]") + glob.glob(f"{DATASET_DIR}/{cls}/*.png")
        counts[cls] = len(img_paths)
        print(f"  • {cls}: {counts[cls]} images")

    # Display sample images
    all_images = []
    for cls in classes:
        imgs = glob.glob(f"{DATASET_DIR}/{cls}/*.[jJ][pP]*[gG]") + glob.glob(f"{DATASET_DIR}/{cls}/*.png")
        all_images.extend([(img, cls) for img in imgs])

    if all_images:
        samples = random.sample(all_images, min(5, len(all_images)))
        fig, axes = plt.subplots(1, len(samples), figsize=(15, 3))
        if len(samples) == 1:
            axes = [axes]
        for ax, (img_path, cls) in zip(axes, samples):
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(cls, fontsize=10)
            ax.axis('off')
        plt.tight_layout()
        plt.show()


## 2. Train MobileNetV2 with Data Augmentation & Fine-Tuning
We use **Transfer Learning** from ImageNet-pretrained MobileNetV2, followed by fine-tuning top convolutional layers.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# 1. Load Train (80%) and Validation (20%) sets
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nDetected {num_classes} classes: {class_names}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 2. Agricultural Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomBrightness(0.12),
    layers.RandomZoom(0.1),
])

# 3. MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n--- 🚀 Training Phase 1: Feature Extraction (10 Epochs) ---")
history_1 = model.fit(train_ds, validation_data=val_ds, epochs=10)

print("\n--- 🚀 Training Phase 2: Fine-Tuning Top Layers (5 Epochs) ---")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
history_2 = model.fit(train_ds, validation_data=val_ds, epochs=5)

# Plot Training Progress
acc = history_1.history['accuracy'] + history_2.history['accuracy']
val_acc = history_1.history['val_accuracy'] + history_2.history['val_accuracy']
loss = history_1.history['loss'] + history_2.history['loss']
val_loss = history_1.history['val_loss'] + history_2.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy', color='teal')
plt.plot(val_acc, label='Val Accuracy', color='orange')
plt.legend(loc='lower right')
plt.title('Accuracy over Epochs')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss', color='teal')
plt.plot(val_loss, label='Val Loss', color='orange')
plt.legend(loc='upper right')
plt.title('Loss over Epochs')
plt.show()

# Confusion Matrix Evaluation
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Greens')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('Rice Disease Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))


## 3. Export to TensorFlow Lite & Auto-Download
Converts the model into optimized `rice_model.tflite` (compatible with Flutter TFLite FFI) and generates `labels.txt`.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image
from google.colab import files

# 1. Save labels.txt in exact class order
with open("labels.txt", "w") as f:
    for cls in class_names:
        f.write(f"{cls}\n")
print("✅ Saved labels.txt with classes:", class_names)

# 2. Convert to TFLite (Keras 3 & TF 2.x compatible)
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
except Exception:
    model.export("saved_model")
    converter = tf.lite.TFLiteConverter.from_saved_model("saved_model")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()

with open("rice_model.tflite", "wb") as f:
    f.write(tflite_model)

tflite_size_mb = os.path.getsize("rice_model.tflite") / (1024 * 1024)
print(f"✅ Successfully created rice_model.tflite! File size: {tflite_size_mb:.2f} MB")

# 3. Quick verification on sample images
interpreter = tf.lite.Interpreter(model_path="rice_model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\n--- 🧪 Testing TFLite model inference on 3 images ---")
for img_path, true_cls in all_images[:3]:
    img = Image.open(img_path).resize((224, 224))
    img_arr = np.array(img, dtype=np.float32)
    img_arr = tf.keras.applications.mobilenet_v2.preprocess_input(img_arr)
    img_arr = np.expand_dims(img_arr, axis=0)
    
    interpreter.set_tensor(input_details[0]['index'], img_arr)
    interpreter.invoke()
    pred = interpreter.get_tensor(output_details[0]['index'])[0]
    pred_idx = np.argmax(pred)
    conf = pred[pred_idx] * 100
    print(f"  • File: {os.path.basename(img_path)} | True: {true_cls} | Predicted: {class_names[pred_idx]} ({conf:.1f}%)")

# 4. Trigger auto-downloads
print("\n📥 Downloading files to your browser...")
files.download("rice_model.tflite")
files.download("labels.txt")
print("🎉 Done! Put these two files in your Flutter project's 'assets/models/' folder.")
